In [1]:
from quantum_fragment_methods.workflow import QFWorkflow
from quantum_fragment_methods.application.embedding import EWF
from quantum_fragment_methods.application.embedding.ewf_pbc import EWF as EWF_PBC
from quantum_fragment_methods.application.solvers.classical_zoo import FCI, CCSD
import vayesta.ewf
import numpy as np

No module named 'dyson'
Dyson required for self-energy calculations


In [2]:
import pyscf
import pyscf.pbc
import pyscf.pbc.scf

cell = pyscf.pbc.gto.Cell()
a = 3.615
cell.atom = "B 0 0 0 ; N %f %f %f" % (a / 4, a / 4, a / 4)
cell.a = np.asarray([[a / 2, a / 2, 0], [0, a / 2, a / 2], [a / 2, 0, a / 2]])
cell.basis = "sto-6g"
cell.output = "pyscf.out"
cell.build()

# Hartree-Fock with k-points
kmesh = [2, 2, 1]
kpts = cell.make_kpts(kmesh)
mf = pyscf.pbc.scf.KRHF(cell, kpts)
mf = mf.density_fit(auxbasis="sto-6g")
mf.kernel()


overwrite output file: pyscf.out


-79.14209837256413

In [3]:
##Full CCSD
emb = vayesta.ewf.EWF(mf, bath_options=dict(bathtype="full"))
# If calling the kernel without initializiation fragmentation and fragments,
# IAO fragmentation and atomic fragments are used automatically.
energy_full_ccsd = emb.kernel()

          |
          |  INITIALIZING EWF
          |  ================
          |      Initial E(mean-field)=     -79.14209837 Ha
          |      Calculated E(HF)=          -79.14209837 Ha
          |      Difference dE=              +0.00000000 Ha ( -0.0%)
          |      n(AO)=   40  n(MO)=   40  n(linear dep.)=    0
          |      Parameters of EWF:
          |      Options(store_eris=True, global_frag_chempot=None, dm_with_frozen=False, bath_options={'bathtype':
          |          'full', 'canonicalize': True, 'occupation_tolerance': 1e-06, 'dmet_threshold': 1e-08, 'rcut': None,
          |          'unit': 'Ang', 'order': None, 'max_order': 20, 'threshold': 1e-08, 'truncation': 'occupation',
          |          'project_dmet_order': 2, 'project_dmet_mode': 'squared-entropy', 'addbuffer': False, 'bathtype_occ':
          |          None, 'bathtype_vir': None, 'rcut_occ': None, 'rcut_vir': None, 'unit_occ': None, 'unit_vir': None,
          |          'order_occ': None, 'or

overwrite output file: f0_pyscf.out


          |      Solving CCSD-equations without initial guess...
          |      T-Diagnostic
          |      ------------
          |        T1= 0.009 (good)  D1= 0.028 (fair)  D2= 0.112 (good)
          |        (T1<0.02: good / D1<0.02: good, D1<0.05: fair / D2<0.15: good, D2<0.18: fair)
          |      Time for CCSD solver: 1 min 2 s
          |  Solving Fragment 1: N
          |  ---------------------
          |      Parameters of RClusterHamiltonian:
          |      RClusterHamiltonian.Options(screening=None, cache_eris=True, match_fock=False)
          |      Parameters of RCCSD_Solver:
          |      RCCSD_Solver.Options(max_cycle=300, conv_tol=None, conv_tol_normt=None, diis_space=None,
          |          diis_start_cycle=None, iterative_damping=None, level_shift=None, init_guess='default',
          |          solve_lambda=True, n_moments=None, sc_mode=0)


overwrite output file: f1_pyscf.out


          |      Solving CCSD-equations without initial guess...
          |      T-Diagnostic
          |      ------------
          |        T1= 0.009 (good)  D1= 0.028 (fair)  D2= 0.112 (good)
          |        (T1<0.02: good / D1<0.02: good, D1<0.05: fair / D2<0.15: good, D2<0.18: fair)
          |      Time for CCSD solver: 1 min 2 s


          |  E(MF)=       -79.14209837 Ha
          |  E(corr)=      -0.09168237 Ha
          |  E(tot)=      -79.23378074 Ha


          |  Total wall time:  2 min 10 s


In [4]:
energy_full_ccsd = -79.23378074 # Bypass running CCSD again
print(energy_full_ccsd)

-79.23378074


In [5]:
## With fragmentation from Vayesta (approx_cumulant = True)
emb = vayesta.ewf.EWF(mf, energy_functional = 'dmet', bath_options=dict(threshold=1e-3))
# If calling the kernel without initializiation fragmentation and fragments,
# IAO fragmentation and atomic fragments are used automatically.
emb.kernel()

          |
          |  INITIALIZING EWF
          |  ================
          |      Initial E(mean-field)=     -79.14209837 Ha
          |      Calculated E(HF)=          -79.14209837 Ha
          |      Difference dE=              +0.00000000 Ha ( -0.0%)
          |      n(AO)=   40  n(MO)=   40  n(linear dep.)=    0
          |      Parameters of EWF:
          |      Options(store_eris=True, global_frag_chempot=None, dm_with_frozen=False, bath_options={'bathtype':
          |          'mp2', 'canonicalize': True, 'occupation_tolerance': 1e-06, 'dmet_threshold': 1e-08, 'rcut': None,
          |          'unit': 'Ang', 'order': None, 'max_order': 20, 'threshold': 0.001, 'truncation': 'occupation',
          |          'project_dmet_order': 2, 'project_dmet_mode': 'squared-entropy', 'addbuffer': False, 'bathtype_occ':
          |          None, 'bathtype_vir': None, 'rcut_occ': None, 'rcut_vir': None, 'unit_occ': None, 'unit_vir': None,
          |          'order_occ': None, 'ord

overwrite output file: f0_pyscf.out


          |      T-Diagnostic
          |      ------------
          |        T1= 0.005 (good)  D1= 0.015 (good)  D2= 0.088 (good)
          |        (T1<0.02: good / D1<0.02: good, D1<0.05: fair / D2<0.15: good, D2<0.18: fair)
          |      Time for CCSD solver: 0.6 s
          |  Solving Fragment 1: N
          |  ---------------------
          |      Parameters of RClusterHamiltonian:
          |      RClusterHamiltonian.Options(screening=None, cache_eris=True, match_fock=False)
          |      Parameters of RCCSD_Solver:
          |      RCCSD_Solver.Options(max_cycle=300, conv_tol=None, conv_tol_normt=None, diis_space=None,
          |          diis_start_cycle=None, iterative_damping=None, level_shift=None, init_guess='default',
          |          solve_lambda=True, n_moments=None, sc_mode=0)
          |      Solving CCSD-equations without initial guess...


overwrite output file: f1_pyscf.out


          |      T-Diagnostic
          |      ------------
          |        T1= 0.006 (good)  D1= 0.016 (good)  D2= 0.087 (good)
          |        (T1<0.02: good / D1<0.02: good, D1<0.05: fair / D2<0.15: good, D2<0.18: fair)
          |      Time for CCSD solver: 0.6 s


          |  E(MF)=       -79.14209837 Ha
          |  E(corr)=      -0.12071458 Ha
          |  E(tot)=      -79.26281296 Ha


          |  Total wall time:  7.6 s


-79.26281295572801

In [6]:
### quantum-fragment-methods EWF-PBC version (approx_cumulant = True)
ewf_embedder = EWF_PBC(
    bath_type='mp2',
    truncation = 1e-3,
    approx_cumulant=True
    # fragment_type is specified in kernel(), not __init__
)

# Create workflow with EWF embedder and IAO fragmentation
workflow_ewf_ccsd = QFWorkflow(
    # geometry="He 0.0 0.0 0.0",
    # basis="def2-svp",
    mf = mf,
    embedder=ewf_embedder,
    fragmentation='atomic',  # Use IAO fragmentation 
    save_path='results_ewf_ccsd/'
)

# Use CCSD for all fragments
workflow_ewf_ccsd.add_solver_rule(
    solver_factory=lambda frag: CCSD(),
    condition=None,  # Applies to all fragments
    priority=0
)
print("Running EWF+CCSD on fragmented system with IAO fragmentation...")
results_ewf_ccsd = workflow_ewf_ccsd.run()

          |
          |  INITIALIZING EWF
          |  ================


Running EWF+CCSD on fragmented system with IAO fragmentation...


          |      Initial E(mean-field)=     -79.14209837 Ha
          |      Calculated E(HF)=          -79.14209837 Ha
          |      Difference dE=              +0.00000000 Ha ( -0.0%)
          |      n(AO)=   40  n(MO)=   40  n(linear dep.)=    0
          |      Parameters of EWF:
          |      Options(store_eris=True, global_frag_chempot=None, dm_with_frozen=False, bath_options={'bathtype':
          |          'mp2', 'canonicalize': True, 'occupation_tolerance': 1e-06, 'dmet_threshold': 1e-08, 'rcut': None,
          |          'unit': 'Ang', 'order': None, 'max_order': 20, 'threshold': 0.001, 'truncation': 'occupation',
          |          'project_dmet_order': 2, 'project_dmet_mode': 'squared-entropy', 'addbuffer': False, 'bathtype_occ':
          |          None, 'bathtype_vir': None, 'rcut_occ': None, 'rcut_vir': None, 'unit_occ': None, 'unit_vir': None,
          |          'order_occ': None, 'order_vir': None, 'max_order_occ': None, 'max_order_vir': None, 'threshold_

          |  Clusters dumped to file '/tmp/ewf_cluster_ezeev1qa.h5'
Solving fragment 0 with CCSD...
  Fragment 0 correlation energy: -0.04749583 Ha
Solving fragment 4 with CCSD...
  Fragment 4 correlation energy: -0.06713252 Ha
reconstruct_energy: approx_cumulant=True

Partitioned Cumulant Energy Reconstruction (PBC-corrected):
  Electronic DMET energy (per cell): -48.05236462 Ha
    1-body fragment contributions:   -55.67586391 Ha
    2-body cumulant contributions:   -0.16475195 Ha
    Global non-cumulant correction:  7.78825125 Ha
  Nuclear repulsion (per cell):      -28.94118556 Ha
  Exxdiv correction:                 -2.26926278 Ha
  Total energy (per cell):           -79.26281296 Ha
  Reference E(MF) (per cell):        -79.14209837 Ha
  Implied E(corr):                   -0.12071458 Ha


In [7]:
### With fragmentation from Vayesta (approx_cumulant = False)
e_tot_approx = emb.get_dmet_energy(part_cumulant=True, approx_cumulant=False)
print(e_tot_approx)

-79.25275436731074


In [8]:
### quantum-fragment-methods EWF-PBC version (approx_cumulant = False)
ewf_embedder = EWF_PBC(
    bath_type='mp2',
    truncation = 1e-3,
    approx_cumulant=False
    # fragment_type is specified in kernel(), not __init__
)

# Create workflow with EWF embedder and IAO fragmentation
workflow_ewf_ccsd = QFWorkflow(
    # geometry="He 0.0 0.0 0.0",
    # basis="def2-svp",
    mf = mf,
    embedder=ewf_embedder,
    fragmentation='atomic',  # Use IAO fragmentation 
    save_path='results_ewf_ccsd/'
)

# Use CCSD for all fragments
workflow_ewf_ccsd.add_solver_rule(
    solver_factory=lambda frag: CCSD(),
    condition=None,  # Applies to all fragments
    priority=0
)
print("Running EWF+CCSD on fragmented system with IAO fragmentation...")
results_ewf_ccsd = workflow_ewf_ccsd.run()

          |


Running EWF+CCSD on fragmented system with IAO fragmentation...


          |  INITIALIZING EWF
          |  ================
          |      Initial E(mean-field)=     -79.14209837 Ha
          |      Calculated E(HF)=          -79.14209837 Ha
          |      Difference dE=              +0.00000000 Ha ( -0.0%)
          |      n(AO)=   40  n(MO)=   40  n(linear dep.)=    0
          |      Parameters of EWF:
          |      Options(store_eris=True, global_frag_chempot=None, dm_with_frozen=False, bath_options={'bathtype':
          |          'mp2', 'canonicalize': True, 'occupation_tolerance': 1e-06, 'dmet_threshold': 1e-08, 'rcut': None,
          |          'unit': 'Ang', 'order': None, 'max_order': 20, 'threshold': 0.001, 'truncation': 'occupation',
          |          'project_dmet_order': 2, 'project_dmet_mode': 'squared-entropy', 'addbuffer': False, 'bathtype_occ':
          |          None, 'bathtype_vir': None, 'rcut_occ': None, 'rcut_vir': None, 'unit_occ': None, 'unit_vir': None,
          |          'order_occ': None, 'order_vir': Non

          |  Clusters dumped to file '/tmp/ewf_cluster_f3ee9bh6.h5'
Solving fragment 0 with CCSD...
  Fragment 0 correlation energy: -0.04749583 Ha
Solving fragment 4 with CCSD...
  Fragment 4 correlation energy: -0.06713252 Ha
reconstruct_energy: approx_cumulant=False

Partitioned Cumulant Energy Reconstruction (PBC-corrected):
  Electronic DMET energy (per cell): -48.04230603 Ha
    1-body fragment contributions:   -55.67586391 Ha
    2-body cumulant contributions:   -0.16522282 Ha
    Global non-cumulant correction:  7.79878071 Ha
  Nuclear repulsion (per cell):      -28.94118556 Ha
  Exxdiv correction:                 -2.26926278 Ha
  Total energy (per cell):           -79.25275437 Ha
  Reference E(MF) (per cell):        -79.14209837 Ha
  Implied E(corr):                   -0.11065599 Ha
